# Metric tree doanh thu: tháng này tăng nhờ thêm khách, mua thường hơn, hay đơn to hơn?

Bộ `online-retail-ii`, sheet **`Year 2010-2011`** (541 910 dòng, 01/12/2010 tới 09/12/2011): cửa hàng quà tặng
online ở Anh, khách lẻ lẫn khách sỉ. Tháng 12/2011 chỉ có dữ liệu tới ngày 9.

**Bối cảnh.** Mỗi tháng CEO hỏi cùng một câu: "Doanh thu tăng (hoặc giảm) là nhờ đâu?". Bạn dựng một metric tree
dùng lại được hằng tháng:

- doanh thu gộp = doanh thu khách có `Customer ID` + doanh thu không có `Customer ID`;
- doanh thu khách có mã = số khách × tần suất (đơn / khách) × AOV (doanh thu / đơn);

rồi phân rã thay đổi doanh thu giữa hai tháng thành đóng góp (GBP) của từng nhánh.

Quy tắc làm sạch dùng chung của track, áp dụng trên từng **dòng**: đổi `Invoice` sang `str`, bỏ hoá đơn bắt đầu
bằng `C`, bỏ dòng `Quantity <= 0` hoặc `Price <= 0`. Doanh thu dòng = `Quantity * Price` (GBP, doanh thu gộp).

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/01-analyst-mindset/exercises/intermediate/01-revenue-metric-tree -q`

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có, đọc sheet 2010-2011 (xlsx: khoảng 30-60 giây)
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

RETAIL = ensure("online-retail-ii")
tx = pd.read_excel(RETAIL / "online_retail_II.xlsx", sheet_name="Year 2010-2011")
print(tx.shape, tx.InvoiceDate.min(), "→", tx.InvoiceDate.max())
print(f"dòng thiếu Customer ID: {tx['Customer ID'].isna().mean():.1%}")
tx.head()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 1. Cây theo tháng

`monthly_tree(tx) -> pd.DataFrame`: mỗi tháng của `InvoiceDate` một dòng, index là ngày đầu tháng
(`DatetimeIndex` tên `month`, tăng dần, chỉ các tháng có dòng hợp lệ). Bảy cột theo thứ tự:

| Cột | Định nghĩa (trên dòng hợp lệ của tháng) |
| --- | --- |
| `revenue` | tổng doanh thu gộp |
| `known_revenue` | doanh thu các dòng có `Customer ID` |
| `guest_revenue` | `revenue - known_revenue` |
| `customers` | số `Customer ID` khác nhau |
| `orders` | số `Invoice` khác nhau của các dòng có `Customer ID` |
| `frequency` | `orders / customers` |
| `aov` | `known_revenue / orders` |

Tháng không có dòng nào có `Customer ID` thì `known_revenue`, `customers`, `orders` bằng 0 (`frequency`, `aov`
là `NaN`). Với cách định nghĩa này, `customers * frequency * aov == known_revenue` đúng tuyệt đối.

Nên viết hàm phụ `_clean(tx)` trả các dòng hợp lệ kèm cột `revenue`.

In [ ]:
def _clean(tx: pd.DataFrame) -> pd.DataFrame:
    """Dòng hợp lệ (Invoice là str), thêm cột revenue = Quantity * Price; không sửa tx."""
    # TODO
    raise NotImplementedError


def monthly_tree(tx: pd.DataFrame) -> pd.DataFrame:
    """Index month; cột revenue, known_revenue, guest_revenue, customers, orders, frequency, aov."""
    # TODO: tháng = InvoiceDate.dt.to_period("M").dt.to_timestamp(); nhánh khách có mã dùng dòng có Customer ID
    raise NotImplementedError

In [ ]:
# explore: cây theo tháng và kiểm tích ba thành phần
tree = monthly_tree(tx)
print(tree.round(2).to_string())
check = tree.customers * tree.frequency * tree.aov - tree.known_revenue
print("lệch lớn nhất của khách × tần suất × AOV so với known_revenue:", float(check.abs().max()))
print(f"tỉ trọng doanh thu có Customer ID cả năm: {tree.known_revenue.sum() / tree.revenue.sum():.1%}")

In [ ]:
# explore: small multiples - bốn nhánh của cây theo tháng (tháng 12/2011 chưa đủ, nét đứt)
fig, axes = plt.subplots(2, 2, figsize=(11, 6), sharex=True)
series = [("customers", "Số khách", 1), ("frequency", "Tần suất (đơn / khách)", 1),
          ("aov", "AOV (GBP / đơn)", 1), ("guest_revenue", "Doanh thu không mã (nghìn GBP)", 1e3)]
for ax, (col, title, scale) in zip(axes.flat, series):
    values = tree[col] / scale
    ax.plot(values.index[:-1], values.iloc[:-1], marker="o", color="#4c78a8")
    ax.plot(values.index[-2:], values.iloc[-2:], ls="--", marker="o", color="#9aa5b1")
    ax.set_ylim(0, None)
    ax.set_title(title, loc="left", fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("Nhánh nào của cây chuyển động theo mùa?", x=0.01, ha="left")
fig.tight_layout()
plt.show()

## 2. Phân rã thay đổi giữa hai tháng

`decompose(tree, m0, m1) -> pd.Series`: đóng góp (GBP) của từng nhánh vào thay đổi doanh thu gộp từ tháng `m0`
sang tháng `m1` (`m0`, `m1` là chuỗi ngày đầu tháng, ví dụ `"2011-10-01"`). Index theo thứ tự
`customers`, `frequency`, `aov`, `guest`; tên Series `contribution`.

- `guest` = `guest_revenue` tháng `m1` trừ tháng `m0`.
- Ba thành phần còn lại phân rã thay đổi của `known_revenue` bằng phân rã log (LMDI, phần 5 lý thuyết). Với
  R là `known_revenue`, đóng góp của thành phần X là

$$
L \ln\frac{X_1}{X_0}, \qquad L = \frac{R_1 - R_0}{\ln R_1 - \ln R_0}
$$

  Khi R1 = R0 (không đổi), dùng L = R0 (giới hạn của công thức).

Bốn đóng góp cộng lại phải **đúng bằng** thay đổi của `revenue`.

In [ ]:
def decompose(tree: pd.DataFrame, m0: str, m1: str) -> pd.Series:
    """Đóng góp GBP của customers, frequency, aov (LMDI trên known_revenue) và guest; tên "contribution"."""
    # TODO: lấy hai dòng bằng tree.loc[pd.Timestamp(m0)], tree.loc[pd.Timestamp(m1)]
    raise NotImplementedError

In [ ]:
# explore: phân rã ba cặp tháng, kiểm tổng
pairs = [("2010-12-01", "2011-01-01"), ("2011-10-01", "2011-11-01"), ("2011-11-01", "2011-12-01")]
for m0, m1 in pairs:
    parts = decompose(tree, m0, m1)
    delta = tree.loc[m1, "revenue"] - tree.loc[m0, "revenue"]
    print(f"{m0[:7]} → {m1[:7]}: Δ {delta:+,.0f} GBP ({tree.loc[m1, 'revenue'] / tree.loc[m0, 'revenue'] - 1:+.1%}); "
          f"tổng đóng góp {parts.sum():+,.0f}")
    print("   ", {k: f"{v:+,.0f}" for k, v in parts.items()})
    print("    thay đổi %:", {k: f"{tree.loc[m1, k] / tree.loc[m0, k] - 1:+.1%}" for k in ["customers", "frequency", "aov", "guest_revenue"]})

In [ ]:
# explore: waterfall tháng 10 → 11/2011
parts = decompose(tree, "2011-10-01", "2011-11-01")
start, end = tree.loc["2011-10-01", "revenue"], tree.loc["2011-11-01", "revenue"]
level_before = start + parts.cumsum().shift(fill_value=0)
labels = ["T10/2011", *parts.index, "T11/2011"]
heights = [start, *parts.abs(), end]
bottoms = [0, *np.minimum(level_before, level_before + parts), 0]
colors = ["#9aa5b1", *["#4c78a8" if v >= 0 else "#e45756" for v in parts], "#9aa5b1"]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(labels, np.array(heights) / 1e3, bottom=np.array(bottoms) / 1e3, color=colors)
ax.set_ylabel("nghìn GBP")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Tháng 10 → 11/2011: nhánh nào đẩy doanh thu?", loc="left")
plt.show()

In [ ]:
# explore: các hoá đơn lớn nhất của tháng 11/2011 và tháng 1/2011 - ai là người mua?
sales = _clean(tx)
inv = sales.groupby("Invoice").agg(revenue=("revenue", "sum"), date=("InvoiceDate", "first"),
                                   customer=("Customer ID", "first"), country=("Country", "first"))
for month in ["2011-11", "2011-01"]:
    print(inv[inv.date.dt.strftime("%Y-%m") == month].nlargest(3, "revenue").round({"revenue": 2}))
print(tx[tx["Invoice"].astype(str).isin(["541431", "C541433"])][["Invoice", "Description", "Quantity", "Price", "Customer ID"]])

## 3. Đọc kết quả, rồi làm trắc nghiệm

Viết cho CEO một câu cho mỗi cặp tháng, theo mẫu "doanh thu đổi X GBP (Y%); nhánh lớn nhất là ... ; câu hỏi
tiếp theo là ...":

1. **Tháng 10 → 11/2011**: doanh thu gộp tăng, nhưng bao nhiêu phần đến từ khách có mã? Nhánh nào âm?
2. **Tháng 12/2010 → 1/2011**: doanh thu khách có mã gần như không đổi. Có thật là "không có gì xảy ra" không?
   Hoá đơn lớn nhất tháng 1 là gì, và chuyện gì xảy ra với nó ngay sau đó?
3. **Tháng 11 → 12/2011**: con số này có nên đưa cho CEO không?

Thử thêm: phân rã bằng cách thay lần lượt từng thành phần (khách trước rồi tần suất rồi AOV, và ngược lại) cho
tháng 10 → 11/2011. Hai thứ tự cho đóng góp của AOV khác nhau bao nhiêu?

Xong thì mở app, làm phần **trắc nghiệm** của bài.